# Simple Logistic Regression — MS Subtype Classification

**Goal:** Predict which MS subtype (RRMS, SPMS, PPMS) a patient has based on clinical and MRI features.

**About Logistic Regression:** A simple linear model that predicts class probabilities using the softmax function. It gives us a baseline and interpretable coefficients.

---

In [ ]:
# Import all the libraries we need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.linear_model import LogisticRegression
# Set seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 13,
    'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3,
})

COLORS = {'RRMS': '#2196F3', 'SPMS': '#FF5722', 'PPMS': '#4CAF50'}
ORDER = ['RRMS', 'SPMS', 'PPMS']

print("All libraries loaded successfully!")

## 2. Loading the Dataset

We load the CSV file and take a quick look at its shape and first few rows.

In [ ]:
df = pd.read_csv('../datasets/ms_dataset.csv')
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

## 3. Exploring the Data

Before building a model, we need to understand our data — check for missing values, see how many patients are in each subtype, and look at feature distributions.

### 3.1 Basic Info

In [ ]:
print("Data types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe().round(2)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values:")
print(missing[missing > 0])
print(f"\nDuplicate rows: {df.duplicated().sum()}")

### 3.2 How Many Patients in Each Subtype?

In [ ]:
counts = df['subtype'].value_counts().reindex(ORDER)
print(counts)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(ORDER, counts, color=[COLORS[s] for s in ORDER], edgecolor='white')
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 2, str(c), ha='center', fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('MS Subtype Distribution')
plt.tight_layout()
plt.show()

### 3.3 Feature Distributions by Subtype

Boxplots help us see which features differ across subtypes — features with clear separation will be helpful for classification.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
ncols = 4
nrows = (len(numeric_cols) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='subtype', y=col, order=ORDER, palette=COLORS, ax=axes[i], fliersize=2)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Features by MS Subtype', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Correlation Heatmap

Shows how features relate to each other. Highly correlated features carry similar information.

In [ ]:
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

Steps:
1. Separate features (X) from target (y)
2. Encode target labels as numbers
3. Split into 80% train / 20% test
4. Fill missing values with the median
5. Scale features to mean=0, std=1

**Important:** We fit the imputer only on training data to prevent data leakage.

In [ ]:
# Separate features and target
feature_cols = [c for c in df.columns if c != 'subtype']
X = df[feature_cols]
y = df['subtype']

# Encode target as numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_
print(f"Classes: {list(class_names)}")

In [ ]:
# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# Fill missing values with median
imputer = SimpleImputer(strategy='median')
X_train_clean = imputer.fit_transform(X_train)
X_test_clean = imputer.transform(X_test)

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_ready = scaler.fit_transform(X_train_clean)
X_test_ready = scaler.transform(X_test_clean)
print("Data scaled (mean=0, std=1)")
print(f"Missing values remaining: 0")

## 5. Building the Model

Logistic Regression with L2 regularization and balanced class weights.

In [ ]:
model = LogisticRegression(
    penalty='l2', solver='lbfgs', max_iter=1000,
    class_weight='balanced', C=1.0, random_state=RANDOM_STATE
)
print("Model created")

## 6. Cross-Validation (5-Fold)

We split training data into 5 parts, train on 4 and test on 1, rotating each time. This tells us how stable the model is.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'accuracy': 'accuracy', 'f1_macro': 'f1_macro', 'f1_weighted': 'f1_weighted'}

cv_results = cross_validate(model, X_train_ready, y_train, cv=cv, scoring=scoring)

print("Cross-Validation Results:")
print(f"  Accuracy:     {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"  Macro F1:     {cv_results['test_f1_macro'].mean():.4f} ± {cv_results['test_f1_macro'].std():.4f}")
print(f"  Weighted F1:  {cv_results['test_f1_weighted'].mean():.4f} ± {cv_results['test_f1_weighted'].std():.4f}")

In [ ]:
# Visualize CV results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = ['Accuracy', 'Macro F1', 'Weighted F1']
means = [cv_results[f'test_{m}'].mean() for m in ['accuracy', 'f1_macro', 'f1_weighted']]
stds = [cv_results[f'test_{m}'].std() for m in ['accuracy', 'f1_macro', 'f1_weighted']]
axes[0].bar(names, means, yerr=stds, capsize=6, color=['#2196F3', '#FF5722', '#4CAF50'], edgecolor='white')
for i, (m, s) in enumerate(zip(means, stds)):
    axes[0].text(i, m + s + 0.01, f'{m:.3f}', ha='center', fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Average CV Scores')

for key, label, c in [('test_accuracy','Accuracy','#2196F3'), ('test_f1_macro','Macro F1','#FF5722'), ('test_f1_weighted','Weighted F1','#4CAF50')]:
    axes[1].plot(range(1,6), cv_results[key], 'o-', label=label, color=c, linewidth=2)
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Score')
axes[1].set_title('Fold-wise Stability')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.suptitle('Logistic Regression — Cross-Validation', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Train on Full Training Set

In [ ]:
model.fit(X_train_ready, y_train)
print("Model trained!")

## 8. Test Set Evaluation

Now we test on data the model has never seen.

In [ ]:
y_pred = model.predict(X_test_ready)
y_proba = model.predict_proba(X_test_ready)

print("Test Results:")
print(f"  Accuracy:     {accuracy_score(y_test, y_pred):.4f}")
print(f"  Macro F1:     {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"  Weighted F1:  {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\n" + classification_report(y_test, y_pred, target_names=class_names, digits=4))

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

### ROC Curves

Closer to the top-left corner = better. AUC of 1.0 is perfect, 0.5 is random guessing.

In [ ]:
y_test_bin = label_binarize(y_test, classes=range(len(class_names)))

fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc(fpr,tpr):.3f})', color=COLORS.get(name))
ax.plot([0,1],[0,1],'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Logistic Regression')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"Overall ROC-AUC: {roc_auc_score(y_test_bin, y_proba, multi_class='ovr', average='macro'):.4f}")

### Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(class_names):
    prec, rec, _ = precision_recall_curve(y_test_bin[:, i], y_proba[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_proba[:, i])
    ax.plot(rec, prec, linewidth=2, label=f'{name} (AP={ap:.3f})', color=COLORS.get(name))
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall — Logistic Regression')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 9. Coefficient Heatmap

Shows how each feature pushes toward or away from each subtype.

In [ ]:
coef_df = pd.DataFrame(model.coef_, index=class_names, columns=feature_cols)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(coef_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Model Coefficients')
plt.tight_layout()
plt.show()

## Conclusion

Logistic Regression gives us a simple interpretable baseline. The coefficients tell us which features matter most for each subtype.